In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalog", "aerolineas_dev")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("storageName", "staerolineasdeveus01")

In [0]:

container = dbutils.widgets.get("container")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storageName = dbutils.widgets.get("storageName")

In [0]:
# definir rutas
path_inicio = f"abfss://{container}@{storageName}.dfs.core.windows.net/"
path_target = f"{catalog}.{schema}."

In [0]:
ruta_csv_fact_vuelos = f"{path_inicio}fact_vuelos.csv"
ruta_csv_fact_pasajeros = f"{path_inicio}fact_pasajeros.csv"
ruta_csv_fact_combustible = f"{path_inicio}fact_combustible.csv"
ruta_csv_fact_ventas_boletos = f"{path_inicio}fact_ventas_boletos.csv"
ruta_csv_fact_equipajes = f"{path_inicio}fact_equipajes.csv"

Ingesta fact para la tabla bbdd_fact_vuelos

In [0]:
vuelos_schema = StructType(fields=[
    StructField("vuelo_id", StringType(), False),
    StructField("aerolinea_id", StringType(), False),
    StructField("aeronave_id", StringType(), False),
    StructField("ruta_id", StringType(), False),
    StructField("numero_vuelo", StringType(), False),
    StructField("fecha_salida_programada_utc", TimestampType(), False),
    StructField("fecha_salida_real_utc", TimestampType(), True),
    StructField("fecha_llegada_programada_utc", TimestampType(), False),
    StructField("fecha_llegada_real_utc", TimestampType(), True),
    StructField("estado_vuelo", StringType(), False),
    StructField("motivo_cancelacion", StringType(), True),
    StructField("terminal_salida", StringType(), False),
    StructField("puerta_embarque", StringType(), True),
    StructField("demora_salida_min", IntegerType(), False),
    StructField("demora_llegada_min", IntegerType(), False),
    StructField("fuente_sistema", StringType(), False)
])

df_fact_vuelos = (
    spark.read.format("csv")
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .option("timeZone", "UTC")
    .schema(vuelos_schema)
    .load(ruta_csv_fact_vuelos)
)

df_fact_vuelos_bronze = df_fact_vuelos.select(
    "vuelo_id",
    "aerolinea_id",
    "aeronave_id",
    "ruta_id",
    "numero_vuelo",
    "fecha_salida_programada_utc",
    "fecha_salida_real_utc",
    "fecha_llegada_programada_utc",
    "fecha_llegada_real_utc",
    "estado_vuelo",
    "motivo_cancelacion",
    "terminal_salida",
    "puerta_embarque",
    "demora_salida_min",
    "demora_llegada_min",
    "fuente_sistema",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_fact_vuelos).alias("archivo_origen")
)

df_fact_vuelos_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_fact_vuelos")

Ingesta fact para la tabla bbdd_fact_pasajeros

In [0]:
pasajeros_schema = StructType(fields=[
    StructField("vuelo_id", StringType(), False),
    StructField("adultos", IntegerType(), False),
    StructField("ninos", IntegerType(), False),
    StructField("infantes_sin_asiento", IntegerType(), False),
    StructField("pasajeros_con_asiento", IntegerType(), False),
    StructField("pasajeros_total_a_bordo", IntegerType(), False),
    StructField("asientos_disponibles", IntegerType(), False),
    StructField("pasajeros_business", IntegerType(), False),
    StructField("pasajeros_economy", IntegerType(), False),
    StructField("transito_conexion", IntegerType(), False),
    StructField("pasajeros_no_show", IntegerType(), False)
])

df_fact_pasajeros = (
    spark.read.format("csv")
    .option("header", True)
    .schema(pasajeros_schema)
    .load(ruta_csv_fact_pasajeros)
)

df_fact_pasajeros_bronze = df_fact_pasajeros.select(
    "vuelo_id",
    "adultos",
    "ninos",
    "infantes_sin_asiento",
    "pasajeros_con_asiento",
    "pasajeros_total_a_bordo",
    "asientos_disponibles",
    "pasajeros_business",
    "pasajeros_economy",
    "transito_conexion",
    "pasajeros_no_show",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_fact_pasajeros).alias("archivo_origen")
)

df_fact_pasajeros_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_fact_pasajeros")

Ingesta fact para la tabla bbdd_fact_combustible

In [0]:
combustible_schema = StructType(fields=[
    StructField("vuelo_id", StringType(), False),
    StructField("tipo_combustible", StringType(), False),
    StructField("combustible_inicial_l", DecimalType(18, 2), False),
    StructField("combustible_cargado_l", DecimalType(18, 2), False),
    StructField("combustible_final_l", DecimalType(18, 2), False),
    StructField("combustible_consumido_l", DecimalType(18, 2), False),
    StructField("precio_usd_l", DecimalType(10, 3), False),
    StructField("costo_carga_usd", DecimalType(18, 2), False),
    StructField("proveedor", StringType(), False),
    StructField("moneda", StringType(), False)
])

df_fact_combustible = (
    spark.read.format("csv")
    .option("header", True)
    .schema(combustible_schema)
    .load(ruta_csv_fact_combustible)
)

df_fact_combustible_bronze = df_fact_combustible.select(
    "vuelo_id",
    "tipo_combustible",
    "combustible_inicial_l",
    "combustible_cargado_l",
    "combustible_final_l",
    "combustible_consumido_l",
    "precio_usd_l",
    "costo_carga_usd",
    "proveedor",
    "moneda",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_fact_combustible).alias("archivo_origen")
)

df_fact_combustible_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_fact_combustible")

Ingesta fact para la tabla bbdd_fact_ventas_boletos

In [0]:
ventas_boletos_schema = StructType(fields=[
    StructField("venta_id", StringType(), False),
    StructField("vuelo_id", StringType(), False),
    StructField("fecha_venta_utc", TimestampType(), False),
    StructField("clase_tarifa", StringType(), False),
    StructField("canal_venta", StringType(), False),
    StructField("boletos_vendidos", IntegerType(), False),
    StructField("boletos_volados", IntegerType(), False),
    StructField("boletos_no_show", IntegerType(), False),
    StructField("tarifa_promedio_usd", DecimalType(18, 2), False),
    StructField("ingreso_bruto_usd", DecimalType(18, 2), False),
    StructField("descuentos_usd", DecimalType(18, 2), False),
    StructField("impuestos_usd", DecimalType(18, 2), False),
    StructField("ingreso_neto_usd", DecimalType(18, 2), False),
    StructField("moneda", StringType(), False)
])

df_fact_ventas_boletos = (
    spark.read.format("csv")
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .option("timeZone", "UTC")
    .schema(ventas_boletos_schema)
    .load(ruta_csv_fact_ventas_boletos)
)

df_fact_ventas_boletos_bronze = df_fact_ventas_boletos.select(
    "venta_id",
    "vuelo_id",
    "fecha_venta_utc",
    "clase_tarifa",
    "canal_venta",
    "boletos_vendidos",
    "boletos_volados",
    "boletos_no_show",
    "tarifa_promedio_usd",
    "ingreso_bruto_usd",
    "descuentos_usd",
    "impuestos_usd",
    "ingreso_neto_usd",
    "moneda",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_fact_ventas_boletos).alias("archivo_origen")
)

df_fact_ventas_boletos_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_fact_ventas_boletos")

Ingesta fact para la tabla bbdd_fact_equipajes

In [0]:
equipajes_schema = StructType(fields=[
    StructField("vuelo_id", StringType(), False),
    StructField("piezas_facturadas", IntegerType(), False),
    StructField("peso_facturado_kg", DecimalType(18, 2), False),
    StructField("piezas_cabina_estimadas", IntegerType(), False),
    StructField("piezas_extraviadas", IntegerType(), False),
    StructField("piezas_danadas", IntegerType(), False),
    StructField("ingreso_equipaje_usd", DecimalType(18, 2), False),
    StructField("tiempo_entrega_promedio_min", IntegerType(), True)
])

df_fact_equipajes = (
    spark.read.format("csv")
    .option("header", True)
    .schema(equipajes_schema)
    .load(ruta_csv_fact_equipajes)
)

df_fact_equipajes_bronze = df_fact_equipajes.select(
    "vuelo_id",
    "piezas_facturadas",
    "peso_facturado_kg",
    "piezas_cabina_estimadas",
    "piezas_extraviadas",
    "piezas_danadas",
    "ingreso_equipaje_usd",
    "tiempo_entrega_promedio_min",
    F.current_timestamp().alias("fecha_ingesta"),
    F.lit(ruta_csv_fact_equipajes).alias("archivo_origen")
)

df_fact_equipajes_bronze.write \
    .mode("overwrite") \
    .insertInto(f"{catalog}.{schema}.bbdd_fact_equipajes")